<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/classify_environmental_sounds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environmental Sound Classification
This notebook develops a deep learning model to classify sounds like dog barks, sirens, and rain using the ESC-50 dataset.

In [20]:
!apt-get install -y libsndfile1
!pip install librosa tensorflow pandas matplotlib requests resampy

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.5 MB/s eta 0:00:00


In [2]:
import os
import requests
import zipfile
import pandas as pd
import numpy as np
import librosa
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Download a subset of ESC-50 or a similar small dataset for demo purposes
# For this example, we will simulate the data loading process assuming the ESC-50 structure
print("Libraries imported. Ready to process audio data.")

Libraries imported. Ready to process audio data.


### Data Preprocessing
We will convert audio files into Mel-spectrograms, which are image-like representations of sound that CNNs can process effectively.

In [17]:
def extract_features(file_name):
    try:
        # Load audio with a fixed duration to handle potential file issues
        audio, sample_rate = librosa.load(file_name, res_type="kaiser_fast", duration=5.0)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        return np.mean(mfccs.T, axis=0)
    except Exception as e:
        # Print the actual error for better debugging
        print(f"Error parsing {file_name}: {e}")
        return None

print("Robust feature extraction function defined.")

Robust feature extraction function defined.


### Model Architecture
We'll use a simple sequential model with Dense layers for the extracted MFCC features.

In [5]:
def create_model(num_labels):
    model = models.Sequential([
        layers.Input(shape=(40,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_labels, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model(3) # Dog bark, Siren, Rain
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,779 (171.01 KB)

 Trainable params: 43,779 (171.01 KB)

 Non-trainable params: 0 (0.00 B)

### Dataset Preparation
We will download the ESC-50 dataset and filter for our target classes (dog, siren, rain).

In [9]:
!git clone https://github.com/karolpiczak/ESC-50.git

# Load metadata
metadata = pd.read_csv('ESC-50/meta/esc50.csv')

# Filter for target classes
target_classes = ['dog', 'siren', 'rain']
filtered_metadata = metadata[metadata['category'].isin(target_classes)].copy()

# Map categories to integers
class_map = {cls: i for i, cls in enumerate(target_classes)}
filtered_metadata['label'] = filtered_metadata['category'].map(class_map)

print(f"Found {len(filtered_metadata)} samples for classes: {target_classes}")

fatal: destination path 'ESC-50' already exists and is not an empty directory.
Found 120 samples for classes: ['dog', 'siren', 'rain']


In [18]:
features = []
labels = []

print("Starting feature extraction...")
# The ESC-50 repository structure puts audio in the 'audio' subfolder
for index, row in filtered_metadata.iterrows():
    file_path = os.path.join('ESC-50', 'audio', row['filename'])
    if os.path.exists(file_path):
        data = extract_features(file_path)
        if data is not None:
            features.append(data)
            labels.append(row['label'])

X = np.array(features)
y = np.array(labels)

if len(X) > 0:
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Extraction complete. Training set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")
else:
    print("Error: No features were extracted. Please check if the dataset was downloaded correctly.")

Starting feature extraction...
Error parsing ESC-50/audio/1-100032-A-0.wav: No module named 'resampy'

This error is lazily reported, having originally occurred in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error parsing ESC-50/audio/1-110389-A-0.wav: No module named 'resampy'

This error is lazily reported, having originally occurred in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error parsing ESC-50/audio/1-17367-A-10.wav: No module named 'resampy'

This error is lazily reported, having originally occurred in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error parsing ESC-50/audio/1-21189-A-10.wav: No module named 'resampy'

This error is lazily reported, having originally occurred in
  File /usr/local/lib/python3.12/dist-packages/lib

### Model Training
Training the model on the extracted features and plotting the accuracy.

In [19]:
if 'X_train' in locals() and len(X_train) > 0:
    print("Training model...")
    history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

    # Plot training & validation accuracy values
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(loc='upper left')
    plt.show()
else:
    print("Training skipped: Data not prepared. Please run the feature extraction cell successfully first.")

Training skipped: Data not prepared. Please run the feature extraction cell successfully first.


### Dataset Preparation
We will download the ESC-50 dataset from GitHub and filter for our target classes.

In [6]:
# Download ESC-50 dataset
!git clone https://github.com/karolpiczak/ESC-50.git

# Load metadata
metadata = pd.read_csv('ESC-50/meta/esc50.csv')

# Filter for target classes
target_classes = ['dog', 'siren', 'rain']
filtered_metadata = metadata[metadata['category'].isin(target_classes)]

# Map categories to integers
class_map = {cls: i for i, cls in enumerate(target_classes)}
filtered_metadata['label'] = filtered_metadata['category'].map(class_map)

print(f"Found {len(filtered_metadata)} samples for classes: {target_classes}")

Cloning into 'ESC-50'...
remote: Enumerating objects: 4199, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 4199 (delta 62), reused 34 (delta 34), pack-reused 4130 (from 1)
Receiving objects: 100% (4199/4199), 878.77 MiB | 30.34 MiB/s, done.
Resolving deltas: 100% (292/292), done.
Updating files: 100% (2011/2011), done.
Found 120 samples for classes: ['dog', 'siren', 'rain']


/tmp/ipykernel_2078/3622845444.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['label'] = filtered_metadata['category'].map(class_map)


In [12]:
features = []
labels = []

print("Starting feature extraction...")
# Ensure the path correctly points to the 'audio' folder inside the cloned repo
for index, row in filtered_metadata.iterrows():
    file_path = os.path.join('ESC-50', 'audio', row['filename'])
    if os.path.exists(file_path):
        data = extract_features(file_path)
        if data is not None:
            features.append(data)
            labels.append(row['label'])

X = np.array(features)
y = np.array(labels)

# Split data only if we have samples
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Extraction complete. Training set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")
else:
    print("No features extracted. Please check if the ESC-50/audio directory contains the .wav files.")

Starting feature extraction...
Error encountered while parsing file: ESC-50/audio/1-100032-A-0.wav
Error encountered while parsing file: ESC-50/audio/1-110389-A-0.wav
Error encountered while parsing file: ESC-50/audio/1-17367-A-10.wav
Error encountered while parsing file: ESC-50/audio/1-21189-A-10.wav
Error encountered while parsing file: ESC-50/audio/1-26222-A-10.wav
Error encountered while parsing file: ESC-50/audio/1-29561-A-10.wav
Error encountered while parsing file: ESC-50/audio/1-30226-A-0.wav
Error encountered while parsing file: ESC-50/audio/1-30344-A-0.wav
Error encountered while parsing file: ESC-50/audio/1-31482-A-42.wav
Error encountered while parsing file: ESC-50/audio/1-31482-B-42.wav
Error encountered while parsing file: ESC-50/audio/1-32318-A-0.wav
Error encountered while parsing file: ESC-50/audio/1-50060-A-10.wav
Error encountered while parsing file: ESC-50/audio/1-54084-A-42.wav
Error encountered while parsing file: ESC-50/audio/1-54958-A-10.wav
Error encountered wh

### Model Training
Now we train the model using the extracted MFCC features.

In [13]:
if 'X_train' in locals():
    print("Starting model training...")
    history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

    # Plot accuracy
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label = 'Val Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.show()
else:
    print("Training skipped: X_train is not defined. Please run the feature extraction cell first.")

Training skipped: X_train is not defined. Please run the feature extraction cell first.
